In [1]:
import pandas as pd

df = pd.read_csv("Reviews.csv")

print(df.shape)
print(df.dtypes)
print(df[['ProductId', 'Score', 'Summary', 'Text']].head(3))

print(df[['Text', 'Summary', 'Score']].isnull().sum())

print(df['Score'].value_counts())

(568454, 10)
Id                        int64
ProductId                   str
UserId                      str
ProfileName                 str
HelpfulnessNumerator      int64
HelpfulnessDenominator    int64
Score                     int64
Time                      int64
Summary                     str
Text                        str
dtype: object
    ProductId  Score                Summary  \
0  B001E4KFG0      5  Good Quality Dog Food   
1  B00813GRG4      1      Not as Advertised   
2  B000LQOCH0      4  "Delight" says it all   

                                                Text  
0  I have bought several of the Vitality canned d...  
1  Product arrived labeled as Jumbo Salted Peanut...  
2  This is a confection that has been around a fe...  
Text        0
Summary    27
Score       0
dtype: int64
Score
5    363122
4     80655
1     52268
3     42640
2     29769
Name: count, dtype: int64


In [2]:
print(df[['Text', 'Summary', 'Score']].isnull().sum())
print("\nScore dağılımı:")
print(df['Score'].value_counts().sort_index())

print(f"\nUnique ürün sayısı: {df['ProductId'].nunique()}")

print("\nÖrnek yorum:")
print(df['Text'].iloc[0][:300])

Text        0
Summary    27
Score       0
dtype: int64

Score dağılımı:
Score
1     52268
2     29769
3     42640
4     80655
5    363122
Name: count, dtype: int64

Unique ürün sayısı: 74258

Örnek yorum:
I have bought several of the Vitality canned dog food products and have found them all to be of good quality. The product looks more like a stew than a processed meat and it smells better. My Labrador is finicky and she appreciates this product better than  most.


In [3]:
import pandas as pd

df = pd.read_csv("Reviews.csv")

# Null Summary'leri doldur
df['Summary'] = df['Summary'].fillna('')

# Her yorum için chunk formatı oluştur
def create_chunk(row):
    sentiment = "olumlu" if row['Score'] >= 4 else ("olumsuz" if row['Score'] <= 2 else "nötr")
    return f"Ürün: {row['ProductId']}\nPuan: {row['Score']}/5 ({sentiment})\nBaşlık: {row['Summary']}\nYorum: {row['Text'][:500]}"

# Pandas 3.x uyumlu stratified sampling — groupby.apply yerine pd.concat
df_sample = pd.concat([
    group.sample(min(len(group), 2000), random_state=42)
    for _, group in df.groupby('Score')
]).reset_index(drop=True)

df_sample['chunk'] = df_sample.apply(create_chunk, axis=1)

print(f"Toplam chunk: {len(df_sample)}")
print(f"\nScore dağılımı (sample):")
print(df_sample['Score'].value_counts().sort_index())
print(f"\nÖrnek chunk:\n{df_sample['chunk'].iloc[0]}")

chunks = df_sample['chunk'].tolist()

Toplam chunk: 10000

Score dağılımı (sample):
Score
1    2000
2    2000
3    2000
4    2000
5    2000
Name: count, dtype: int64

Örnek chunk:
Ürün: B000O160KE
Puan: 1/5 (olumsuz)
Başlık: Sweet & Low without the cancer.
Yorum: If you like the (bitter) taste of Sweet & Low, get this. If you don't, don't. Couldn't get through one cup of coffee (and I only used 1/2 of the tiny packet).<br /><br />I'm gonna give "Stevia Extract in the Raw" a try. It's made by the folks at "Sugar in the Raw." (And, no, I don't work for their company.) Here's what they claim:<br /><br />"Stevia Extract In The Raw gets its delicious, natural sweetness from Rebiana (aka Reb-A) -- an extract from the Stevia plant. This extract is the sweetest 


In [4]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
import pickle

# Model yükle — İngilizce için hızlı ve kaliteli
model = SentenceTransformer('all-MiniLM-L6-v2')
print(f'Model yüklendi: {model.get_sentence_embedding_dimension()} boyutlu vektörler')

# Embedding oluştur (10_000 chunk için ~1-2 dk)
print('Embedding oluşturuluyor...')
embeddings = model.encode(chunks, batch_size=64, show_progress_bar=True, convert_to_numpy=True)

print(f'Embedding shape: {embeddings.shape}')

# FAISS index oluştur (cosine similarity için normalize et)
faiss.normalize_L2(embeddings)
index = faiss.IndexFlatIP(embeddings.shape[1])  # Inner Product = cosine (normalize sonrası)
index.add(embeddings)

print(f'FAISS index oluşturuldu: {index.ntotal} vektör')

# Kaydet
faiss.write_index(index, 'reviews.index')
with open('chunks.pkl', 'wb') as f:
    pickle.dump(chunks, f)

print('Index ve chunk listesi diske kaydedildi.')

/Users/wert/PycharmProjects/AmazonProductReviews/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9397.11it/s]


BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model yüklendi: 384 boyutlu vektörler
Embedding oluşturuluyor...


...

Batches:  95%|█████████▍| 149/157 [00:17<00:00, 17.74it/s]

Batches:  97%|█████████▋| 152/157 [00:17<00:00, 19.06it/s]

Batches:  99%|█████████▊| 155/157 [00:17<00:00, 19.85it/s]

Batches: 100%|██████████| 157/157 [00:17<00:00,  8.87it/s]

Embedding shape: (10000, 384)
FAISS index oluşturuldu: 10000 vektör
Index ve chunk listesi diske kaydedildi.


In [5]:
def search(query, k=5):
    """Doğal dil sorgusu ile en alakalı yorumları bul."""
    q_vec = model.encode([query], convert_to_numpy=True)
    faiss.normalize_L2(q_vec)
    distances, indices = index.search(q_vec, k)

    print(f"\n🔍 Sorgu: '{query}'")
    print('=' * 60)
    for rank, (idx, score) in enumerate(zip(indices[0], distances[0]), 1):
        print(f"\n[{rank}] Benzerlik: {score:.3f}")
        print(chunks[idx])
        print('-' * 60)

# Demo sorgular
search('dog food with good quality and taste')
search('terrible product, waste of money')
search('best coffee ever')


🔍 Sorgu: 'dog food with good quality and taste'

[1] Benzerlik: 0.761
Ürün: B002LANN56
Puan: 3/5 (nötr)
Başlık: Woof! Woof! Woof, woof, woof! (... or, apparently tastes great, but...)
Yorum: Normally my daughter's Pomeranian is fed a diet of <a href="http://www.amazon.com/gp/product/B000W5U5H6">Taste of the Wild Dry Dog Food, Hi Prairie Canine Formula with Roasted Bison & Venison, 15-Pound Bag</a>, or another of their variety's of dry dog food.  Thanks to some previous recommendations (a nice chart/web site that compared the various dog food products and rated them based on meat content, grain and other by-product inclusion, etc.), we'd settled on that as one of the better quality p
------------------------------------------------------------

[2] Benzerlik: 0.747
Ürün: B002LANN56
Puan: 2/5 (olumsuz)
Başlık: Low Quality Dog Food
Yorum: Unless you are already feeding your dog a low end dog food, Chef Michael's Grilled Sirloin is not something I'd recommend feeding to your pet.<br /><br

In [6]:
import pickle

# df_sample'dan metadata kaydet (chunk başına Score ve ProductId)
metadata = df_sample[['Score', 'ProductId']].to_dict('records')

with open('metadata.pkl', 'wb') as f:
    pickle.dump(metadata, f)

print(f'Metadata kaydedildi: {len(metadata)} kayıt')
print(f'Örnek: {metadata[0]}')
print(f'Score dağılımı: {df_sample["Score"].value_counts().sort_index().to_dict()}')

Metadata kaydedildi: 10000 kayıt
Örnek: {'Score': 1, 'ProductId': 'B000O160KE'}
Score dağılımı: {1: 2000, 2: 2000, 3: 2000, 4: 2000, 5: 2000}


In [ ]:
# Gradio uygulaması app.py olarak kaydedildi.
# Terminalde çalıştır:
#   .venv/bin/python app.py
#
# Veya doğrudan buradan başlat:
import subprocess, sys
subprocess.Popen([sys.executable, 'app.py'])
print('Uygulama başlatılıyor → http://127.0.0.1:7860')